<!-- SPDX-License-Identifier: CC-BY-4.0 -->
<!-- Copyright Contributors to the Egeria project. -->

# 05 Conformance and drift on a BIDS-shaped tree

Coco Pharmaceuticals' imaging team keeps study data in BIDS layout (Brain Imaging Data Structure). A data-management team wants to know whether a
tree follows its declared layout, and what changed since the last survey.

This notebook generates a synthetic BIDS-shaped tree with empty data files (no real data). It surveys the tree with the layout recognizer and the
`bids-dataset` profile and reports conformance in the form "97% conformant; 12 findings". It then injects deviations, surveys again, and reports
drift in the form "3 new since last survey".

Findings have a kind: `structure` (missing or unexpected), `naming` (misnamed) or `content-type` (three fields of `dataset_description.json`).

The notebook runs locally with Python and PyYAML only. Nothing is fetched. Nothing is written to Egeria.
The last cells, for the Resource Explorer survey, Portal curation and Egeria, are marked "Phase 3 — not run".

## Set up

The recognizer lives in `layout-profiles/` of this repository, two folders up from this notebook. The work folder is a fresh temporary folder.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

LAYOUT = (Path.cwd() / ".." / ".." / "layout-profiles").resolve()
assert (LAYOUT / "recognizer" / "recognize.py").is_file(), "run this notebook from its own folder in a checkout of egeria-workspaces"
sys.path.insert(0, str(LAYOUT))
sys.path.insert(0, str(LAYOUT / "samples"))

import bids_gen
from recognizer import conformance, drift
from recognizer import manifests
from recognizer.layout_profile import load_profile
from recognizer.recognize import Recognizer

PROFILE = load_profile("bids-dataset", LAYOUT / "profiles")
WORK = Path(tempfile.mkdtemp(prefix="nb05-"))
print(f"profile {PROFILE.profile_id} {PROFILE.version}; work folder made")


def generate(name, seed=1, subjects=6, sessions=2, spec=None):
    """Write a generated tree to WORK/name: dataset/, paths.tsv and truth.jsonl."""
    files, truth = bids_gen.generate(seed, subjects, sessions, bids_gen.load_spec(spec) if spec else [])
    bids_gen.write_outputs(files, truth, WORK / name)
    return truth


def survey(name, paths=None, root=None):
    """Recognize one tree (names and sizes, plus the three dataset_description.json fields) and write its conformance report."""
    paths = paths or WORK / name / "paths.tsv"
    root = root or WORK / name / "dataset"
    out = WORK / f"{name}-survey"
    Recognizer(PROFILE, blob_source=manifests.FileSource(str(root))).run(Path(paths), out)
    return conformance.write(out), out

## 1. Generate a clean tree

Six subjects, two sessions each, with the four checked datatypes `anat`, `func`, `dwi` and `fmap`. The seed picks the tasks, runs and field-map layout.

In [ ]:
generate("clean")
rows = (WORK / "clean" / "paths.tsv").read_text().splitlines()
print(len(rows), "files, for example:")
for row in rows[:4] + rows[4:40:9]:
    print("  ", row.split("\t")[0])
print(json.loads((WORK / "clean" / "dataset" / "dataset_description.json").read_text()))

## 2. Survey it and report conformance

The score is conformant checked items over checked items. Every entry of a folder that the profile closes, every required path and every checked
field of `dataset_description.json` is one item.

In [ ]:
clean, clean_out = survey("clean")
print(conformance.headline(clean))
print("checked:", clean["checked"], "| not checked:", clean["not_checked_files"], "files")

## 3. Inject deviations and survey again

`samples/specs/drift-a.yaml` asks the generator for one deviation of each structure and naming class and two content-type ones. The generator writes
what it injected to `truth.jsonl`; the survey does not see that file.

The score is also split: BIDS (what the standard asks for) and study consistency (every subject has every session and datatype folder seen,
a regular study's protocol, which BIDS does not require).

In [ ]:
truth = generate("changed", spec=LAYOUT / "samples" / "specs" / "drift-a.yaml")
changed, changed_out = survey("changed")
print(conformance.headline(changed))
for kind, statuses in changed["findings_by_kind_status"].items():
    print(f"  {kind}: {statuses}")
cats = changed["by_category"]
print(f"BIDS {cats['standard']['score']:.1%}, study consistency {cats['study-consistency']['score']:.1%}")

## 4. Report drift since the last survey

Findings are compared by `finding_id`, which is made from the profile, the rule and the path only. A missing file's finding has the EXPECTED path, so
it keeps its id across surveys.

In [ ]:
d = drift.write(clean_out, changed_out)
print(drift.headline(d))
for f in d["new"]:
    print(f"  {f['kind']:<12} {f['status']:<13} {f['path']}  ({f['rule_id']}{', ' + f['reason'] if f.get('reason') else ''})")

## 5. The UC6 questions, answered from the stored outputs

Only `conformance.json` and `drift.json` are read here.

In [ ]:
c = json.loads((changed_out / "conformance.json").read_text())
dj = json.loads((changed_out / "drift.json").read_text())
by_status = {}
for f in c["findings"]:
    by_status.setdefault(f["status"], []).append(f["path"])

print("Does the tree match its declared layout?", "yes" if c["findings_total"] == 0 else f"no: {conformance.headline(c)}")
print("Which folders or files are missing?      ", by_status.get("missing", []))
print("Which are unexpected?                    ", by_status.get("unexpected", []))
print("Which are wrongly named?                 ", by_status.get("misnamed", []))
print("Which description fields are wrong?      ", [f"{f['rule_id']} ({f['status']})" for f in c["findings"] if f["kind"] == "content-type"])
print("Has it drifted since the last survey?    ", drift.headline(dj))
print("Over the allowed count?                  ", "not checked in v0 (no count rules in bids-dataset 0.1.1)")

## 6. Check the answer against what was injected

The key is exact, because every deviation was injected on purpose.

In [ ]:
injected = sorted((t["path"], t["kind"], t["status"], t["rule"]) for t in truth)
found = sorted((f["path"], f["kind"], f["status"], f["rule_id"]) for f in d["new"])
print(f"{len(injected)} injected, {len(found)} new findings, equal: {injected == found}")
assert injected == found

## 7. Fix the tree and survey again

The clean tree is written again. Every deviation is resolved, with the same finding ids.

In [ ]:
generate("fixed")
fixed, fixed_out = survey("fixed")
back = drift.write(changed_out, fixed_out)
print(conformance.headline(fixed), "|", drift.headline(back))
assert {f["finding_id"] for f in back["resolved"]} == {f["finding_id"] for f in d["new"]}

## 8. A real dataset: ds000117, by names only

`layout-profiles/samples/ds000117/` holds the file list (names and sizes) of the bids-examples dataset ds000117 (CC0) and its
`dataset_description.json`, nothing else. Its findings are REPORTED, not scored: they come from where this profile is stricter than BIDS
(see the README's Known limits): 5 `dwi` folders that some subjects lack, study-consistency findings, and none against the standard. Folders
outside the profile's subset (derivatives, stimuli, sub-emptyroom, meg) are counted, not checked.

In [ ]:
from collections import Counter

ds = LAYOUT / "samples" / "ds000117"
real, real_out = survey("ds000117", paths=ds / "paths.tsv", root=ds)
print(conformance.headline(real), f"| {real['not_checked_files']} of {real['total_files']} files not checked")
cats = real["by_category"]
print(f"BIDS {cats['standard']['score']:.1%}, study consistency {cats['study-consistency']['score']:.1%}")
reasons = Counter((f["kind"], f["status"], f.get("reason", "")) for f in real["findings"])
for (kind, status, reason), n in reasons.most_common():
    print(f"  {n:>4}  {kind} {status} {reason}")

## Phase 3 — not run

In Phase 3 the Resource Explorer will run the same survey on a governed tree. The Portal will show the score and the findings list for
accept-or-fix decisions. The findings could be recorded in Egeria.

None of this runs here: it needs a survey window and the owner's agreement, and Egeria types for layout profiles wait for Gate 2.
The cells below only show the steps; they do nothing.

In [ ]:
# Phase 3 — not run: the Resource Explorer survey of a governed tree with the bids-dataset profile attached.
PHASE3 = False
if PHASE3:
    raise RuntimeError("Phase 3 — not run")
print("Phase 3 — not run: Resource Explorer survey")

In [ ]:
# Phase 3 — not run: Portal curation (the conformance score, the findings list, accept-or-fix decisions).
if PHASE3:
    raise RuntimeError("Phase 3 — not run")
print("Phase 3 — not run: Portal curation")

In [ ]:
# Phase 3 — not run: record the findings in Egeria (the layout profile types wait for Gate 2).
if PHASE3:
    raise RuntimeError("Phase 3 — not run")
print("Phase 3 — not run: Egeria")

## Clean up

In [ ]:
import shutil

shutil.rmtree(WORK)
print("work folder removed")